# Multimodal AI: Vision-Language Models

**Module 05 | Notebook 5 of 5**

## Introduction

In this notebook, we'll explore **multimodal AI models** that can understand and reason about both text and images. These models, like GPT-4o and Gemini, represent a major advancement in AI capabilities.

### What You'll Learn

- What are vision-language models and how they work
- Image captioning and visual question answering
- Document understanding and OCR
- Building multimodal applications with **GPT-4o-mini** (OpenAI)
- Building multimodal applications with **Gemini 2.0 Flash** (Google AI Studio — free tier)
- Real-world use cases

### Prerequisites

- Understanding of LLMs (Module 05, Notebooks 1–4)
- OpenAI API key (for Sections 1–5)
- Gemini API key — free at [aistudio.google.com](https://aistudio.google.com) (for Section 6)
- Python basics

---

## Understanding Multimodal AI

### What Are Vision-Language Models?

**Vision-language models** can process and understand multiple modalities (text + images) simultaneously.

```
Traditional LLM:     Text Input  →  [Model]  →  Text Output
Multimodal Model:    Text + Image  →  [Model]  →  Text Output
```

### Leading Multimodal Models (2026)

| Model | Provider | Strengths | Use Cases |
|-------|----------|-----------|-----------|
| **GPT-4o** | OpenAI | Fast, cost-effective, vision | Production apps |
| **GPT-4o-mini** | OpenAI | Cheapest multimodal option | High-volume tasks |
| **Gemini** | Google | Long context, video | Document analysis |
| **Claude** | Anthropic | Safety, accuracy, vision | Enterprise use |
| **LLaVA** | Open-source | Customizable | Research, fine-tuning |

### Key Capabilities

1. **Image Understanding**: Describe what's in an image
2. **Visual Q&A**: Answer questions about images
3. **OCR**: Extract text from images
4. **Document Analysis**: Understand charts, diagrams, tables
5. **Visual Reasoning**: Solve problems requiring visual understanding

---

## Setup and Installation

In [ ]:
!uv pip install -Uq openai pillow matplotlib requests

In [ ]:
import os
import base64
from openai import OpenAI
from PIL import Image
import matplotlib.pyplot as plt
import requests
from io import BytesIO

# --- API Key Setup ---
# Google Colab: Set your key in Colab Secrets (key icon in sidebar)
# Local/VSCode: Create a .env file with OPENAI_API_KEY=sk-...

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found. Set it in Colab Secrets or a .env file.")

client = OpenAI(api_key=api_key)
print("✅ Setup complete!")

---

## 1. Image Captioning

Let's start with the simplest task: describing what's in an image.

In [ ]:
# Helper function to display images
def display_image(url, title="Image"):
    headers = {"User-Agent": "Mozilla/5.0"}  
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    img = Image.open(BytesIO(response.content))
    print(f"{title} ({img.size[0]}x{img.size[1]})")
    display(img)

In [ ]:
# Example 1: Image from URL
image_url = "https://upload.wikimedia.org/wikipedia/commons/1/10/Raccoon_procyon_lotor.jpg"

# Display the image
display_image(image_url, "Input Image")

# Analyze with GPT-4o-mini (multimodal model)
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "What is in this image? Provide a detailed description."},
                {"type": "image_url", "image_url": {"url": image_url}}
            ]
        }
    ],
    max_tokens=300
)

caption = response.choices[0].message.content
print("\n📝 Image Caption:")
print(caption)

---

## 2. Visual Question Answering

Ask specific questions about images.

In [ ]:
# Ask specific questions about the same image
questions = [
    "What animal is this?",
    "What is the animal doing?",
    "What is the setting or environment?",
    "What time of day does it appear to be?"
]

print("\n🔍 Visual Question Answering:\n")

for question in questions:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": question},
                    {"type": "image_url", "image_url": {"url": image_url}}
                ]
            }
        ],
        max_tokens=100
    )
    
    answer = response.choices[0].message.content
    print(f"Q: {question}")
    print(f"A: {answer}\n")

---

## 3. Working with Local Images

Analyze images from your local filesystem using base64 encoding.

In [ ]:
def encode_image(image_path):
    """Encode a local image to base64"""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def analyze_local_image(image_path, prompt):
    """Analyze a local image with a multimodal model"""
    # Encode image
    base64_image = encode_image(image_path)
    
    # Get file extension
    ext = image_path.split('.')[-1].lower()
    mime_type = f"image/{ext}" if ext in ['png', 'jpg', 'jpeg', 'gif', 'webp'] else "image/jpeg"
    
    # Analyze
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": f"data:{mime_type};base64,{base64_image}"}}
                ]
            }
        ],
        max_tokens=300
    )
    
    return response.choices[0].message.content

# Example usage (uncomment and provide your own image path)
# image_path = "path/to/your/image.jpg"
# display_image(image_path, "Local Image")
# result = analyze_local_image(image_path, "Describe this image in detail")
# print(result)

---

## 4. Document Understanding and OCR

Extract and understand text from images, charts, and documents.

In [ ]:
# Example: Analyze a chart or document
# Using a sample chart image
chart_url = "https://upload.wikimedia.org/wikipedia/commons/6/6c/Bar_Chart.png"

display_image(chart_url, "Chart/Document")

# Extract information from the chart
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": """Analyze this chart and provide:
                1. What type of chart is this?
                2. What data does it show?
                3. What are the key insights?"""},
                {"type": "image_url", "image_url": {"url": chart_url}}
            ]
        }
    ],
    max_tokens=400
)

print("\nChart Analysis:")
print(response.choices[0].message.content)

---

## 5. Multi-Image Analysis

Compare and analyze multiple images together.

In [ ]:
# Compare two images
image1_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg"
image2_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/1200px-Cat_November_2010-1a.jpg"

headers = {"User-Agent": "Mozilla/5.0"}   # ← fix

# Display both images
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for idx, url in enumerate([image1_url, image2_url]):
    response = requests.get(url, headers=headers)   # ← fix
    img = Image.open(BytesIO(response.content))
    axes[idx].imshow(img)
    axes[idx].axis('off')
    axes[idx].set_title(f"Image {idx+1}")

plt.tight_layout()
plt.show()

# Compare the images
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Compare these two images. What are the similarities and differences?"},
                {"type": "image_url", "image_url": {"url": image1_url}},
                {"type": "image_url", "image_url": {"url": image2_url}}
            ]
        }
    ],
    max_tokens=300
)

print("\n🔄 Image Comparison:")
print(response.choices[0].message.content)


---

## 6. Gemini Vision (Google AI Studio)

[Google Gemini](https://ai.google.dev/) models are powerful multimodal models with a **free tier** available through Google AI Studio. In this section we use the **native `google-genai` SDK** — giving us access to Gemini's full feature set rather than the OpenAI-compatibility shim seen in Notebook 1.

| Feature | GPT-4o-mini (above) | Gemini 2.0 Flash (this section) |
|---|---|---|
| **Free tier** | ❌ Pay-per-token | ✅ Google AI Studio |
| **Image input** | URL or base64 string | Bytes via `Part.from_bytes()` |
| **Context window** | 128K tokens | 1M tokens |
| **SDK** | `openai` Python client | `google-genai` SDK |

> **Get your free key**: Visit [aistudio.google.com](https://aistudio.google.com) → Create API key. Store it as `GEMINI_API_KEY` in Colab Secrets or your `.env` file.

In [ ]:
!uv pip install -q google-genai

In [ ]:
from google import genai
from google.genai import types as genai_types

# Option 1 — Google Colab: Load API key from Colab Secrets
# from google.colab import userdata
# gemini_api_key = userdata.get('GEMINI_API_KEY')

# Option 2 — Local (VSCode / Jupyter): Load API key from .env file
# from dotenv import load_dotenv
# load_dotenv()
# gemini_api_key = os.getenv("GEMINI_API_KEY")

# gemini_api_key = os.getenv("GEMINI_API_KEY")

gemini_client = genai.Client(api_key=gemini_api_key)
print("✅ Gemini client ready!")

In [ ]:
def analyze_image_gemini(image_url_or_bytes, prompt, mime_type="image/jpeg"):
    """Analyze an image with Gemini Vision.
    
    Accepts a URL string or raw image bytes.
    Gemini's native API takes bytes via Part.from_bytes(), unlike OpenAI's URL/base64 format.
    """
    if isinstance(image_url_or_bytes, str):
      headers = {"User-Agent": "Mozilla/5.0"} 
      image_data = requests.get(image_url_or_bytes, headers=headers).content
    else:
      image_data = image_url_or_bytes

    result = gemini_client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=[
            genai_types.Part.from_bytes(data=image_data, mime_type=mime_type),
            genai_types.Part.from_text(text=prompt),
        ],
    )
    return result.text


# Demo: Image Captioning with Gemini (same raccoon image from Section 1)
display_image(image_url, "Input Image")
description = analyze_image_gemini(image_url, "Describe this image in detail.")
print("📝 Gemini Caption:")
print(description)

### Gemini Visual Question Answering

Gemini can answer targeted questions about images just like GPT-4o-mini. Below we ask the same questions from Section 2 so you can compare the responses side-by-side.

In [ ]:
# The same questions from Section 2 — now answered by Gemini
questions = [
    "What animal is this?",
    "What is the animal doing?",
    "What is the setting or environment?",
]

print("🔍 Gemini Visual Q&A:\n")

for question in questions:
    answer = analyze_image_gemini(image_url, question)
    print(f"Q: {question}")
    print(f"A: {answer}\n")

### GPT-4o-mini vs Gemini 2.0 Flash: Side-by-Side

Both models excel at standard vision tasks. Here are the practical differences to guide your choice:

| Feature | GPT-4o-mini | Gemini 2.0 Flash |
|---|---|---|
| **Cost** | Pay-per-token | Free tier (AI Studio) |
| **Image input API** | `image_url` content type | `Part.from_bytes()` |
| **Context window** | 128K tokens | 1M tokens |
| **SDK** | `openai` Python client | `google-genai` SDK |
| **Multi-image** | ✅ | ✅ |

> **Bottom line:** Both are strong choices. Use Gemini when you want a free tier for prototyping; use GPT-4o-mini when you're already within an OpenAI workflow.

---

## 7. Real-World Use Cases

### Use Case 1: Accessibility - Image Alt Text Generation

In [ ]:
def generate_alt_text(image_url):
    """Generate accessible alt text for images"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": """Generate concise, descriptive alt text for this image 
                    suitable for screen readers. Focus on the main subject and important details.
                    Keep it under 125 characters."""},
                    {"type": "image_url", "image_url": {"url": image_url}}
                ]
            }
        ],
        max_tokens=50
    )
    return response.choices[0].message.content

# Example
test_image = "https://commons.wikimedia.org/wiki/Category:Procyon_lotor#/media/File:Raccoon_in_Central_Park_(35264).jpg"
alt_text = generate_alt_text(test_image)
print(f"Generated Alt Text: {alt_text}")

### Use Case 2: E-commerce - Product Analysis

In [ ]:
def analyze_product_image(image_url):
    """Analyze product images for e-commerce"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": """Analyze this product image and provide:
                    1. Product category
                    2. Key features visible
                    3. Suggested product title
                    4. Suggested product description (2-3 sentences)"""},
                    {"type": "image_url", "image_url": {"url": image_url}}
                ]
            }
        ],
        max_tokens=300
    )
    return response.choices[0].message.content

# Example (use any product image URL)
# product_analysis = analyze_product_image("YOUR_PRODUCT_IMAGE_URL")
# print(product_analysis)

### Use Case 3: Content Moderation

In [ ]:
def moderate_image(image_url):
    """Check if image contains inappropriate content"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": """Analyze this image for content moderation.
                    Is it safe for work? Does it contain:
                    - Violence
                    - Adult content
                    - Hate symbols
                    - Other inappropriate content
                    
                    Respond with: SAFE or UNSAFE, followed by a brief explanation."""},
                    {"type": "image_url", "image_url": {"url": image_url}}
                ]
            }
        ],
        max_tokens=150
    )
    return response.choices[0].message.content

# Example with safe image
moderation_result = moderate_image(test_image)
print(f"Moderation Result:\n{moderation_result}")

---

## 8. Best Practices

### Image Quality Guidelines

| Aspect | Recommendation | Why |
|--------|----------------|-----|
| **Resolution** | 512-2048px | Balance quality and cost |
| **Format** | PNG, JPEG, WebP | Widely supported |
| **File Size** | < 20MB | API limits |
| **Clarity** | High quality, well-lit | Better results |

### Prompt Engineering for Vision

**Good Vision Prompts:**
- "Describe the main objects in this image"
- "What is the person in the image doing?"
- "Extract all text visible in this document"
- "Compare the two products shown"

**Avoid:**
- Vague questions ("Tell me about this")
- Asking for information not visible in the image
- Expecting perfect OCR on low-quality images

### Cost Optimization

```python
# Use gpt-4o-mini for most tasks (cheapest multimodal option)
model = "gpt-4o-mini"

# Resize images before sending
# Lower resolution = lower cost

# Use detail parameter
# "low" = faster and cheaper
# "high" = better quality
{"type": "image_url", "image_url": {"url": url, "detail": "low"}}
```

---

## 9. Hands-On Exercise

### Your Turn!

Build a multimodal application for one of these scenarios:

1. **Recipe Analyzer**: Upload food images and get recipe suggestions
2. **Document Scanner**: Extract and summarize information from documents
3. **Visual Search**: Find similar products based on image
4. **Accessibility Tool**: Generate detailed descriptions for visually impaired users

In [ ]:
# Exercise: Recipe Analyzer
def analyze_food_image(image_url):
    """Analyze food images and suggest recipes"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": """Analyze this food image and provide:
                    1. What dish is this?
                    2. Main ingredients visible
                    3. Estimated cuisine type
                    4. Brief recipe suggestion"""},
                    {"type": "image_url", "image_url": {"url": image_url}}
                ]
            }
        ],
        max_tokens=400
    )
    return response.choices[0].message.content

# Try with a food image
# food_url = "YOUR_FOOD_IMAGE_URL"
# result = analyze_food_image(food_url)
# print(result)

---

## Summary

### What We Learned

✅ **Multimodal AI**: Understanding vision-language models  
✅ **Image Captioning**: Describing images automatically (GPT-4o-mini + Gemini)  
✅ **Visual Q&A**: Answering questions about images  
✅ **Document Understanding**: OCR and chart analysis  
✅ **Multi-Image Analysis**: Comparing multiple images  
✅ **Gemini Vision**: Free-tier multimodal with native `google-genai` SDK  
✅ **Real-World Applications**: Accessibility, e-commerce, moderation  

### Key Takeaways

1. **Multimodal models unlock new use cases** beyond text-only AI
2. **GPT-4o-mini** is cost-effective for production vision tasks
3. **Gemini 2.0 Flash** offers a **free tier** and 1M token context — ideal for prototyping and long-document analysis
4. **Image quality matters** for better results
5. **Prompt engineering** applies to vision tasks too
6. **Two SDKs, same concept**: OpenAI's `image_url` format vs Gemini's `Part.from_bytes()` — different syntax, identical capability

### Provider Comparison

| Provider | Model | Free Tier | Image Input | Best For |
|---|---|---|---|---|
| OpenAI | `gpt-4o-mini` | ❌ | URL or base64 | Production, existing OpenAI workflows |
| Google | `gemini-2.0-flash` | ✅ | `Part.from_bytes()` | Prototyping, long context, free tier |

### Comparison: Text-Only vs Multimodal

| Task | Text-Only LLM | Multimodal Model |
|------|---------------|------------------|
| Describe a product | ❌ Needs text description | ✅ Analyzes image directly |
| Extract data from chart | ❌ Can't see chart | ✅ Reads and interprets |
| Answer "What's in this photo?" | ❌ Impossible | ✅ Natural capability |
| Generate image captions | ❌ Needs pre-generated text | ✅ Creates from image |

### Next Steps

- Experiment with **video understanding** (Gemini supports video frames)
- Build **multimodal RAG** systems (text + image retrieval)
- Try **Claude Vision** for safety-critical applications

### Resources

- [OpenAI Vision Guide](https://platform.openai.com/docs/guides/vision)
- [Google Gemini Vision](https://ai.google.dev/gemini-api/docs/vision)
- [Google AI Studio (free Gemini key)](https://aistudio.google.com)
- [Claude Vision](https://docs.anthropic.com/claude/docs/vision)